In [0]:
import json
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs, compute

In [0]:


def create_or_update_pipeline(config, replace=False):
    w = WorkspaceClient()

    with open(config, 'r') as f:
        config_dict = json.load(f)
    
    existing_job = next((j for j in w.jobs.list() if j.settings.name == config_dict['name']), None)

    if existing_job and replace:
        print(f"Overwriting job: {config_dict['name']}")
        w.jobs.delete(job_id=existing_job.job_id)
        existing_job = None

    git_conf = jobs.GitSource(
        git_url=config_dict['git_source']['git_url'],
        git_provider=jobs.GitProvider(config_dict['git_source']['git_provider']),
        git_branch=config_dict['git_source']['git_branch']
    )

    job_tasks = [
        jobs.Task(
            task_key=t['task_key'],
            run_if=jobs.RunIf(t['run_if']),
            notebook_task=jobs.NotebookTask(
                notebook_path=t['notebook_task']['notebook_path'],
                base_parameters=t['notebook_task'].get('base_parameters', {}),
                # Ensure the source is GIT as per your config
                source=jobs.Source(t['notebook_task']['source'])
            ),
            depends_on=[jobs.TaskDependency(task_key=d['task_key']) for d in t.get('depends_on', [])]
        ) for t in config_dict['tasks']
    ]

    run_as_conf = None
    if 'run_as' in config_dict:
        run_as_conf = jobs.JobRunAs(
            service_principal_name=config_dict['run_as'].get('service_principal_name')
        )

    job_settings = jobs.JobSettings(
        name=config_dict['name'],
        tasks=job_tasks,
        git_source=git_conf,
        max_concurrent_runs=config_dict.get('max_concurrent_runs', 1),
        email_notifications=jobs.JobEmailNotifications(
            no_alert_for_skipped_runs=config_dict['email_notifications']['no_alert_for_skipped_runs']
        ),
        queue=jobs.QueueSettings(enabled=config_dict['queue']['enabled'])
    )

    if existing_job:
        print(f"Updating: {config_dict['name']}")
        w.jobs.reset(job_id=existing_job.job_id, new_settings=job_settings)
        return existing_job.job_id
    else:
        print(f"Creating fresh Serverless job: {config_dict['name']}")

        created_job = w.jobs.create(
            name=job_settings.name,
            tasks=job_settings.tasks,
            git_source=job_settings.git_source,
            run_as=run_as_conf,
            max_concurrent_runs=job_settings.max_concurrent_runs,
            email_notifications=job_settings.email_notifications,
            queue=job_settings.queue
        )
        return f"Successfully created job: {job_settings.name}"